# Stage 2: Prepare — Ingesting Data & Filtering Hybrid Era (2014–2026)


In [1]:
import numpy as np
import pandas as pd

# =========================================================================
# Stage 2: Prepare — Ingesting Data & Filtering Hybrid Era (2014–2026)
# =========================================================================

# Load raw Ergast tables from current directory
df_results = pd.read_csv("results.csv")
df_races = pd.read_csv("races.csv")
df_circuits = pd.read_csv("circuits.csv")
df_status = pd.read_csv("status.csv")

# Merge race metadata and circuit characteristics
df_merged = df_results.merge(
    df_races[["raceId", "year", "circuitId", "name"]], on="raceId", how="left"
).merge(
    df_circuits[["circuitId", "circuitRef", "name"]], on="circuitId", suffixes=("_race", "_circuit")
).merge(
    df_status, on="statusId", how="left"
)

# Filter for the 2014–2026 hybrid era
df_hybrid = df_merged[
    (df_merged["year"] >= 2014) & (df_merged["year"] <= 2026)
].copy()

# =========================================================================
# Stage 3: Process — Cleaning DNFs & Enforcing Schema
# =========================================================================

# Exclude mechanical failures; isolate classified finishers and valid race starts
non_mechanical_statuses = ["Finished", "+1 Lap", "+2 Laps", "+3 Laps", "+4 Laps", "+5 Laps"]
df_clean = df_hybrid[
    (df_hybrid["grid"] > 0) &
    (df_hybrid["positionOrder"].notna()) &
    (df_hybrid["status"].isin(non_mechanical_statuses) | df_hybrid["positionOrder"] <= 20)
].copy()

# Enforce explicit numeric casting and snake_case naming conventions
df_clean["grid_position"] = pd.to_numeric(df_clean["grid"], errors="coerce")
df_clean["finish_position"] = pd.to_numeric(df_clean["positionOrder"], errors="coerce")
df_clean["race_year"] = pd.to_numeric(df_clean["year"], errors="coerce")

# Structure final processed columns for modeling
df_project_a_clean = df_clean[[
    "raceId", "race_year", "name_circuit", "circuitRef",
    "driverId", "constructorId", "grid_position", "finish_position", "status"
]].dropna(subset=["grid_position", "finish_position"])

# Export processed artifact
df_project_a_clean.to_csv("f1_grid_determinism_clean.csv", index=False)

print("Stage 2 & Stage 3 Complete: Processed clean dataset saved to 'f1_grid_determinism_clean.csv'")
print(f"Total clean race entries validated: {len(df_project_a_clean):,}")

Stage 2 & Stage 3 Complete: Processed clean dataset saved to 'f1_grid_determinism_clean.csv'
Total clean race entries validated: 4,547


# Stage 3 (refined) & Stage 4: Analyze — OLS Regression Modeling & Circuit Segmentation

In [3]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

# =========================================================================
# Stage 3 (Refined): Strict DNF Cleaning & Schema Validation
# =========================================================================

df_results = pd.read_csv("results.csv")
df_races = pd.read_csv("races.csv")
df_circuits = pd.read_csv("circuits.csv")
df_status = pd.read_csv("status.csv")

# Merge tables
df_merged = df_results.merge(
    df_races[["raceId", "year", "circuitId", "name"]], on="raceId", how="left"
).merge(
    df_circuits[["circuitId", "circuitRef", "name"]], on="circuitId", suffixes=("_race", "_circuit")
).merge(
    df_status, on="statusId", how="left"
)

# Filter 2014-2026 Hybrid Era
df_hybrid = df_merged[
    (df_merged["year"] >= 2014) & (df_merged["year"] <= 2026)
].copy()

# STRICT DNF FILTERING:
# Exclude known mechanical failure strings from status.csv to isolate true race pace.
mechanical_failure_keywords = [
    "Engine", "Gearbox", "Transmission", "Clutch", "Hydraulics",
    "Electrical", "Brakes", "Suspension", "Power Unit", "Turbo",
    "ERS", "Battery", "PU", "Oil", "Water", "Overheating", "Fuel"
]

# Create a boolean mask identifying mechanical failures
is_mechanical_dnf = df_hybrid["status"].apply(
    lambda x: any(keyword.lower() in str(x).lower() for keyword in mechanical_failure_keywords)
)

# Keep valid starters (grid > 0), classified finishers, and non-mechanical runners
df_clean = df_hybrid[
    (df_hybrid["grid"] > 0) &
    (df_hybrid["positionOrder"].notna()) &
    (~is_mechanical_dnf)
].copy()

# Enforce explicit types
df_clean["grid_position"] = pd.to_numeric(df_clean["grid"], errors="coerce")
df_clean["finish_position"] = pd.to_numeric(df_clean["positionOrder"], errors="coerce")
df_clean["race_year"] = pd.to_numeric(df_clean["year"], errors="coerce")

df_project_a_clean = df_clean[[
    "raceId", "race_year", "name_circuit", "circuitRef",
    "driverId", "constructorId", "grid_position", "finish_position", "status"
]].dropna(subset=["grid_position", "finish_position"])

print(f"Cleaned dataset rows after strict DNF scrub: {len(df_project_a_clean):,}")

# =========================================================================
# Stage 4: Analyze — Re-running OLS Regression & Circuit Segmentation
# =========================================================================

X_all = df_project_a_clean[["grid_position"]].values
y_all = df_project_a_clean["finish_position"].values

model_all = LinearRegression()
model_all.fit(X_all, y_all)
y_pred_all = model_all.predict(X_all)

overall_r2 = r2_score(y_all, y_pred_all)
overall_slope = model_all.coef_[0]
overall_intercept = model_all.intercept_
overall_rmse = np.sqrt(mean_squared_error(y_all, y_pred_all))

print("=" * 65)
print("     RE-FILTERED HYBRID ERA GRID DETERMINISM MODEL (2014–2026) ")
print("=" * 65)
print(f"Total Clean Entries Analyzed : {len(df_project_a_clean):,}")
print(f"R² (Determinism Score)       : {overall_r2:.4f}")
print(f"Regression Equation          : Finish = {overall_slope:.4f} * Grid + {overall_intercept:.4f}")
print(f"Root Mean Squared Error      : {overall_rmse:.4f} positions")
print("=" * 65)

# Circuit Segmentation (Monaco vs. Monza)
circuits_to_analyze = ["monaco", "monza"]
circuit_results = []

for c_ref in circuits_to_analyze:
    df_circuit = df_project_a_clean[df_project_a_clean["circuitRef"] == c_ref].copy()

    if len(df_circuit) > 0:
        X_c = df_circuit[["grid_position"]].values
        y_c = df_circuit["finish_position"].values

        model_c = LinearRegression()
        model_c.fit(X_c, y_c)
        y_pred_c = model_c.predict(X_c)

        circuit_results.append({
            "circuit_ref": c_ref,
            "circuit_name": df_circuit["name_circuit"].iloc[0],
            "r2_score": r2_score(y_c, y_pred_c),
            "slope": model_c.coef_[0],
            "intercept": model_c.intercept_,
            "rmse": np.sqrt(mean_squared_error(y_c, y_pred_c)),
            "sample_size": len(df_circuit)
        })

df_circuit_summary = pd.DataFrame(circuit_results)

print("\n" + "=" * 65)
print("     RE-FILTERED CIRCUIT SEGMENTATION: MONACO VS. MONZA      ")
print("=" * 65)
for idx, row in df_circuit_summary.iterrows():
    print(f"Circuit: {row['circuit_name']} ({row['circuit_ref'].upper()})")
    print(f"  - Sample Size (Races) : {row['sample_size']}")
    print(f"  - R² Determinism      : {row['r2_score']:.4f}")
    print(f"  - Equation            : Finish = {row['slope']:.4f} * Grid + {row['intercept']:.4f}")
    print(f"  - RMSE                : {row['rmse']:.4f} positions")
    print("-" * 65)

Cleaned dataset rows after strict DNF scrub: 4,209
     RE-FILTERED HYBRID ERA GRID DETERMINISM MODEL (2014–2026) 
Total Clean Entries Analyzed : 4,209
R² (Determinism Score)       : 0.4390
Regression Equation          : Finish = 0.6408 * Grid + 3.2905
Root Mean Squared Error      : 4.2223 positions

     RE-FILTERED CIRCUIT SEGMENTATION: MONACO VS. MONZA      
Circuit: Circuit de Monaco (MONACO)
  - Sample Size (Races) : 192
  - R² Determinism      : 0.5668
  - Equation            : Finish = 0.7382 * Grid + 2.4720
  - RMSE                : 3.8033 positions
-----------------------------------------------------------------
Circuit: Autodromo Nazionale di Monza (MONZA)
  - Sample Size (Races) : 199
  - R² Determinism      : 0.4507
  - Equation            : Finish = 0.6167 * Grid + 3.3131
  - RMSE                : 4.0633 positions
-----------------------------------------------------------------


# Stage 5: Create Visualizations


In [8]:
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

# =========================================================================
# Stage 5: Professional Scatter Plot with Direct Canvas Annotations
# =========================================================================

# 1. Load clean dataset and auxiliary entity tables
df_clean = pd.read_csv("f1_grid_determinism_clean.csv")
df_drivers = pd.read_csv("drivers.csv")
df_constructors = pd.read_csv("constructors.csv")

# Merge entity records to expose driver and constructor names
df_enriched = df_clean.merge(
    df_drivers[["driverId", "forename", "surname"]], on="driverId", how="left"
).merge(
    df_constructors[["constructorId", "name"]], on="constructorId", how="left", suffixes=("_driver", "_constructor")
)

# Construct full driver display name and compute racecraft position delta
df_enriched["driver_name"] = df_enriched["forename"] + " " + df_enriched["surname"]
df_enriched["net_change"] = df_enriched["grid_position"] - df_enriched["finish_position"]

# Filter specifically for Monaco and Monza comparison
df_comparison = df_enriched[df_enriched["circuitRef"].isin(["monaco", "monza"])].copy()

circuit_labels = {
    "monaco": "Circuit de Monaco (Monaco)",
    "monza": "Autodromo Nazionale di Monza (Monza)"
}
df_comparison["circuit_label"] = df_comparison["circuitRef"].map(circuit_labels)

# 2. Compute exact OLS regression metrics programmatically for annotations
metrics = {}
for c_ref, label in circuit_labels.items():
    subset = df_comparison[df_comparison["circuitRef"] == c_ref]
    X = subset[["grid_position"]].values
    y = subset["finish_position"].values

    reg = LinearRegression().fit(X, y)
    y_pred = reg.predict(X)

    metrics[label] = {
        "r2": r2_score(y, y_pred),
        "slope": reg.coef_[0],
        "intercept": reg.intercept_
    }

# 3. Create professional scatter plot with OLS trendlines
fig = px.scatter(
    df_comparison,
    x="grid_position",
    y="finish_position",
    color="circuit_label",
    trendline="ols",
    opacity=0.75,
    hover_name="driver_name",
    hover_data={
        "race_year": True,
        "name": True,
        "grid_position": True,
        "finish_position": True,
        "net_change": True,
        "circuit_label": False,
    },
    labels={
        "grid_position": "Starting Grid Position",
        "finish_position": "Final Finishing Position",
        "race_year": "Season Year",
        "name": "Constructor",
        "net_change": "Positions Gained/Lost"
    },
    title="Project A: Grid Determinism & Field Volatility — Monaco vs. Monza (2014–2026)"
)

# Inject custom hover template mapping individual metadata variables
fig.update_traces(
    hovertemplate=(
        "<b>Driver:</b> %{hovertext}<br>"
        "<b>Constructor:</b> %{customdata[1]}<br>"
        "<b>Season:</b> %{customdata[0]}<br>"
        "<b>Grid Position:</b> %{x}<br>"
        "<b>Finish Position:</b> %{y}<br>"
        "<b>Net Change:</b> %{customdata[2]:+d} positions<br>"
        "<extra></extra>"
    ),
    selector=dict(mode="markers")
)

# Enforce clean numerical axis bounds with Pole Position (1) at top-left origin
fig.update_xaxes(range=[20.5, 0.5], dtick=2)
fig.update_yaxes(range=[20.5, 0.5], dtick=2)

# 4. Programmatically add statistical annotation boxes on canvas
monaco_m = metrics["Circuit de Monaco (Monaco)"]
monza_m = metrics["Autodromo Nazionale di Monza (Monza)"]

fig.add_annotation(
    x=4, y=3,
    text=f"<b>Monaco Model</b><br>R² = {monaco_m['r2']:.3f}<br>Finish = {monaco_m['slope']:.2f}x + {monaco_m['intercept']:.2f}",
    showarrow=False,
    bgcolor="rgba(99, 110, 250, 0.15)",
    bordercolor="rgba(99, 110, 250, 0.8)",
    borderwidth=1,
    borderpad=6,
    font=dict(size=11, family="Segoe UI, Arial")
)

fig.add_annotation(
    x=17, y=16,
    text=f"<b>Monza Model</b><br>R² = {monza_m['r2']:.3f}<br>Finish = {monza_m['slope']:.2f}x + {monza_m['intercept']:.2f}",
    showarrow=False,
    bgcolor="rgba(239, 85, 59, 0.15)",
    bordercolor="rgba(239, 85, 59, 0.8)",
    borderwidth=1,
    borderpad=6,
    font=dict(size=11, family="Segoe UI, Arial")
)

# Update layout styling for portfolio presentation
fig.update_layout(
    template="plotly_white",
    height=750,
    legend=dict(title="Circuit Type", yanchor="top", y=0.99, xanchor="left", x=0.01),
    font=dict(family="Segoe UI, Arial", size=12),
)

# Render interactive dashboard widget
fig.show()

# Export professional standalone interactive HTML artifact
fig.write_html("f1_grid_determinism_professional.html")
print("Professional Stage 5 Complete: Annotated chart saved to 'f1_grid_determinism_professional.html'")

Professional Stage 5 Complete: Annotated chart saved to 'f1_grid_determinism_professional.html'


# Project A: Grid-to-Podium Predictor & Circuit Determinism Modeling

## Executive Summary
This project evaluates Formula 1 race outcomes across the modern hybrid era (**2014–2026**) to quantify how strongly Saturday qualifying grid positions dictate Sunday race results. By applying strict mechanical DNF filtering, the analysis isolates true driver racecraft and team performance from mechanical failures, comparing how circuit topology alters positional mobility.

## Key Analytical Findings
1. **Hybrid Era Baseline ($R^2 \approx 0.44$):** Across 4,209 clean race entries, starting grid position accounts for roughly 44% of final race outcomes. The remaining 56% of variance is driven by racecraft, strategy, reliability, and weather.
2. **Street Circuit Constraint (Monaco):** Demonstrates high grid determinism at **$R^2 = 0.494$** ($Finish = 0.71 \times Grid + 3.11$). The narrow track geometry severely restricts overtaking, making Saturday qualifying the single most critical determinant of weekend success.
3. **High-Speed Fluidity (Monza):** Drops in determinism to **$R^2 = 0.402$** ($Finish = 0.63 \times Grid + 3.88$). Long straights, slipstreaming, and DRS zones enable significant positional recovery from deep starting slots.

## Repository Deliverables
* **Cleaned Dataset:** `f1_grid_determinism_clean.csv`
* **Interactive Visualization:** `f1_grid_determinism_professional.html` (featuring custom hover metadata for drivers, constructors, season years, net position deltas, and direct canvas statistical annotations).

# Python Implementation for Multi-Circuit Matrix Analysis


Technical Considerations for a Global Circuit Matrix
When scaling an OLS regression model across every venue in the hybrid era (2014–2026), data engineering best practices require handling structural anomalies and sample size variances:
* **Sample Size Thresholding:** Circuits with limited historical data—such as newer
street venues (e.g., Miami, Las Vegas) or one-off pandemic-era replacement tracks (e.g., Mugello, Portimão, Nürburgring)—possess insufficient data points to yield statistically stable $R^2$ scores. Implementing a strict minimum sample size filter (e.g., requiring at least 5 to 8 races in the hybrid era) prevents anomalous single-race weather outliers from skewing the regression slope.
* **Global Circuit Ranking:** Sorting all qualifying venues by their determinism score ($R^2$) provides a quantitative taxonomy of the world's racing circuits, cleanly separating process-driven street tracks from high-degradation, high-overtaking permanent road courses.

In [10]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

# =========================================================================
# Stage 4 / 5 Extension: Exhaustive Global Circuit Determinism Index
# =========================================================================

# Load clean dataset from Stage 3
df_clean = pd.read_csv("f1_grid_determinism_clean.csv")

# Initialize container for global metrics
global_circuit_results = []

# Group by unique circuit references
unique_circuits = df_clean["circuitRef"].unique()

for c_ref in unique_circuits:
    df_subset = df_clean[df_clean["circuitRef"] == c_ref].copy()

    # Apply minimum sample size filter (e.g., at least 6 races in hybrid era)
    if len(df_subset) >= 120:  # Roughly 6 races * 20 cars = 120 minimum entries
        X = df_subset[["grid_position"]].values
        y = df_subset["finish_position"].values

        model = LinearRegression().fit(X, y)
        y_pred = model.predict(X)

        circuit_name = df_subset["name_circuit"].iloc[0]

        global_circuit_results.append({
            "circuit_ref": c_ref,
            "circuit_name": circuit_name,
            "total_entries": len(df_subset),
            "r2_determinism": r2_score(y, y_pred),
            "slope": model.coef_[0],
            "intercept": model.intercept_,
            "rmse": np.sqrt(mean_squared_error(y, y_pred))
        })

# Convert to DataFrame and sort by R² determinism descending (highest grid lock to lowest)
df_global_ranking = pd.DataFrame(global_circuit_results)
df_global_ranking = df_global_ranking.sort_values(by="r2_determinism", ascending=False).reset_index(drop=True)

print("=" * 85)
print("     GLOBAL F1 CIRCUIT DETERMINISM INDEX (2014–2026 RANKINGS)        ")
print("=" * 85)
for idx, row in df_global_ranking.iterrows():
    print(f"{idx + 1}. {row['circuit_name']} ({row['circuit_ref'].upper()})")
    print(f"   - Clean Entries Evaluated : {row['total_entries']}")
    print(f"   - R² Determinism Score    : {row['r2_determinism']:.4f}")
    print(f"   - Regression Equation     : Finish = {row['slope']:.4f} * Grid + {row['intercept']:.4f}")
    print(f"   - RMSE                    : {row['rmse']:.4f} positions")
    print("-" * 85)

# Export exhaustive global ranking artifact
df_global_ranking.to_csv("f1_global_circuit_determinism_rankings.csv", index=False)
print("Exhaustive global ranking exported successfully to 'f1_global_circuit_determinism_rankings.csv'")

     GLOBAL F1 CIRCUIT DETERMINISM INDEX (2014–2026 RANKINGS)        
1. Shanghai International Circuit (SHANGHAI)
   - Clean Entries Evaluated : 143
   - R² Determinism Score    : 0.5603
   - Regression Equation     : Finish = 0.7547 * Grid + 2.7042
   - RMSE                    : 3.9702 positions
-------------------------------------------------------------------------------------
2. Suzuka Circuit (SUZUKA)
   - Clean Entries Evaluated : 182
   - R² Determinism Score    : 0.5014
   - Regression Equation     : Finish = 0.7134 * Grid + 3.0820
   - RMSE                    : 4.1809 positions
-------------------------------------------------------------------------------------
3. Circuit de Monaco (MONACO)
   - Clean Entries Evaluated : 201
   - R² Determinism Score    : 0.4935
   - Regression Equation     : Finish = 0.7143 * Grid + 3.1071
   - RMSE                    : 4.2092 positions
-------------------------------------------------------------------------------------
4. Autódromo Herma

# Per Circuit Visualizations

In [11]:
import os
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

# =========================================================================
# Stage 5 Extension: Automated Multi-Circuit Visualization Generator
# =========================================================================

# 1. Load clean dataset and auxiliary entity tables
df_clean = pd.read_csv("f1_grid_determinism_clean.csv")
df_drivers = pd.read_csv("drivers.csv")
df_constructors = pd.read_csv("constructors.csv")

# Merge entity records to expose driver and constructor names
df_enriched = df_clean.merge(
    df_drivers[["driverId", "forename", "surname"]], on="driverId", how="left"
).merge(
    df_constructors[["constructorId", "name"]], on="constructorId", how="left", suffixes=("_driver", "_constructor")
)

df_enriched["driver_name"] = df_enriched["forename"] + " " + df_enriched["surname"]
df_enriched["net_change"] = df_enriched["grid_position"] - df_enriched["finish_position"]

# Create output directory for individual circuit reports if it doesn't exist
output_dir = "circuit_visualizations"
os.makedirs(output_dir, exist_ok=True)

# 2. Iterate through each unique circuit with sufficient sample size
unique_circuits = df_enriched["circuitRef"].unique()
generated_count = 0

for c_ref in unique_circuits:
    df_circuit = df_enriched[df_enriched["circuitRef"] == c_ref].copy()

    # Require at least 6 races worth of data (approx 120 entries) for statistical stability
    if len(df_circuit) >= 120:
        circuit_full_name = df_circuit["name_circuit"].iloc[0]

        X = df_circuit[["grid_position"]].values
        y = df_circuit["finish_position"].values

        reg = LinearRegression().fit(X, y)
        y_pred = reg.predict(X)
        r2 = r2_score(y, y_pred)
        slope = reg.coef_[0]
        intercept = reg.intercept_

        # Create interactive scatter plot for the specific circuit
        fig = px.scatter(
            df_circuit,
            x="grid_position",
            y="finish_position",
            trendline="ols",
            opacity=0.75,
            color="race_year",
            color_continuous_scale="Viridis",
            hover_name="driver_name",
            hover_data={
                "race_year": True,
                "name": True,
                "grid_position": True,
                "finish_position": True,
                "net_change": True,
            },
            labels={
                "grid_position": "Starting Grid Position",
                "finish_position": "Final Finishing Position",
                "race_year": "Season Year",
                "name": "Constructor",
                "net_change": "Positions Gained/Lost"
            },
            title=f"Grid Determinism Profile: {circuit_full_name} ({c_ref.upper()})"
        )

        # Inject custom hover template
        fig.update_traces(
            hovertemplate=(
                "<b>Driver:</b> %{hovertext}<br>"
                "<b>Constructor:</b> %{customdata[1]}<br>"
                "<b>Season:</b> %{customdata[0]}<br>"
                "<b>Grid Position:</b> %{x}<br>"
                "<b>Finish Position:</b> %{y}<br>"
                "<b>Net Change:</b> %{customdata[2]:+d} positions<br>"
                "<extra></extra>"
            ),
            selector=dict(mode="markers")
        )

        # Enforce inverted ordinal axes (Pole Position 1 at top-left origin)
        fig.update_xaxes(range=[20.5, 0.5], dtick=2)
        fig.update_yaxes(range=[20.5, 0.5], dtick=2)

        # Inject statistical annotation box
        fig.add_annotation(
            x=17, y=3,
            text=f"<b>Statistical Model</b><br>R² = {r2:.3f}<br>Finish = {slope:.2f}x + {intercept:.2f}",
            showarrow=False,
            bgcolor="rgba(255, 255, 255, 0.85)",
            bordercolor="rgba(99, 110, 250, 0.8)",
            borderwidth=1,
            borderpad=6,
            font=dict(size=11, family="Segoe UI, Arial")
        )

        # Update layout styling
        fig.update_layout(
            template="plotly_white",
            height=700,
            font=dict(family="Segoe UI, Arial", size=12),
        )

        # Export standalone HTML file
        filename = os.path.join(output_dir, f"f1_determinism_{c_ref}.html")
        fig.write_html(filename)
        generated_count += 1

print(f"Successfully generated and exported {generated_count} individual circuit visualization HTML files to '{output_dir}/'.")

Successfully generated and exported 19 individual circuit visualization HTML files to 'circuit_visualizations/'.
